# 30 – Modeling & Evaluation

Train baseline and tuned models on the feature-selected datasets, evaluate using MAE/R², and produce a Kaggle submission.

- Inputs: `20_x_train.csv`, `20_y_train.csv`, `20_x_val.csv`, `20_y_val.csv`, `20_x_test.csv`.
- Outputs: `30_submission_YYYYMMDD_HHMM.csv` in `../data/submissions/`.

Note: All model selection uses the validation split; only the final chosen model is trained on train+val before generating test predictions.


<a id="top"></a>
## Table of Contents

1. [Setup & Imports](#sec-1-imports)
2. [Load Dataset](#sec-2-load)
3. [Metrics & Utilities](#sec-3-utils)
4. [Baseline Models](#sec-4-baselines)
5. [Model Comparison](#sec-5-compare)
6. [Hyperparameter Tuning](#sec-6-tuning)
7. [Submission File](#sec-7-submission)
8. [Kaggle Submission](#sec-8-kaggle)
9. [Error Analysis](#sec-9-error)

[Back to top](#top)


<a id="sec-1-imports"></a>
## 1. Import Libraries


In [60]:
# Imports
%pip install python-dotenv
import os
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
from pathlib import Path
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.feature_selection import RFE
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from datetime import datetime
from sklearn.metrics import mean_absolute_error, r2_score

Note: you may need to restart the kernel to use updated packages.


<a id="sec-2-load"></a>
## 2. Load Dataset

Load feature-selected splits from step 20 (train/val/test). Ensure shapes match and targets align with their features.

In [61]:
# Define paths
data_dir = "../data/"
encoded_dir = os.path.join(data_dir, "feature_selection")

# Load feature-selected datasets (from step 20)
x_train = pd.read_csv(os.path.join(encoded_dir, "20_x_train.csv"))
y_train = pd.read_csv(os.path.join(encoded_dir, "20_y_train.csv")).squeeze()
x_val = pd.read_csv(os.path.join(encoded_dir, "20_x_val.csv"))
y_val = pd.read_csv(os.path.join(encoded_dir, "20_y_val.csv")).squeeze()
x_test = pd.read_csv(os.path.join(encoded_dir, "20_x_test.csv"))

# Put carID back as index, so it is not in the features anymore
x_train.set_index('CarID', inplace=True)
x_val.set_index('CarID', inplace=True)
x_test.set_index('CarID', inplace=True)
y_train.set_index('CarID', inplace=True)
y_val.set_index('CarID', inplace=True)

# Sanity checks
print("Encoded datasets successfully loaded!")
print(f"x_train shape: {x_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"x_val shape:   {x_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"x_test shape:  {x_test.shape}")


Encoded datasets successfully loaded!
x_train shape: (55850, 25)
y_train shape: (55850, 1)
x_val shape:   (18617, 25)
y_val shape:   (18617, 1)
x_test shape:  (32567, 25)


<a id="sec-3-utils"></a>
## 3. Metrics & Utilities


In [62]:
def evaluate_regression(y_true, y_pred):
    """Compute common regression metrics.
    Returns a dict with MAE and R² for concise reporting.
    """
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred))
    return {"MAE": mae, "R2": r2}

# Helper: print nicely formatted metrics for a model
def report_model(name, y_true, y_pred):
    m = evaluate_regression(y_true, y_pred)
    print(f"[{name}]  MAE: {m['MAE']:,.2f} | R²: {m['R2']:.4f}")
    return m


<a id="sec-4-baselines"></a>
## 4. Baseline Models


In [63]:
# Baseline Model — Linear Regression

lin = LinearRegression()
lin.fit(x_train, y_train)

y_val_pred_lin = lin.predict(x_val)
metrics_lin = report_model("LinearRegression (baseline)", y_val, y_val_pred_lin)


[LinearRegression (baseline)]  MAE: 2,796.09 | R²: 0.7944


In [64]:
# Baseline Model — Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(x_train, y_train)
y_val_pred_ridge = ridge.predict(x_val)
metrics_ridge = report_model("Ridge (baseline)", y_val, y_val_pred_ridge)


[Ridge (baseline)]  MAE: 2,795.88 | R²: 0.7944


In [66]:
# Baseline Model — Lasso

lasso = Lasso(alpha=0.1, max_iter=10000)
lasso.fit(x_train, y_train)
y_val_pred_lasso = lasso.predict(x_val)
metrics_lasso = report_model("Lasso (baseline)", y_val, y_val_pred_lasso)

[Lasso (baseline)]  MAE: 2,795.82 | R²: 0.7944


c:\Users\jdiog\anaconda3\envs\anaconda-2025.06-py3.11\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.613e+08, tolerance: 5.284e+08
  model = cd_fast.enet_coordinate_descent(


In [68]:
# Baseline Model - ElasticNet

elasticnet = ElasticNet(alpha=0.1, l1_ratio=0.5)
elasticnet.fit(x_train, y_train)
y_val_pred_elasticnet = elasticnet.predict(x_val)
metrics_elasticnet = report_model("ElasticNet (baseline)", y_val, y_val_pred_elasticnet)


[ElasticNet (baseline)]  MAE: 2,960.63 | R²: 0.7683


In [69]:
# Baseline Model — RandomForest
y_train_flat = y_train.iloc[:, 0].values
y_val_flat = y_val.iloc[:, 0].values

rf = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)
rf.fit(x_train, y_train_flat)

y_val_pred_rf = rf.predict(x_val)
metrics_rf = report_model("RandomForest (baseline)", y_val_flat, y_val_pred_rf)


[RandomForest (baseline)]  MAE: 1,317.90 | R²: 0.9392


In [70]:
# Baseline Model - DecisionTree

dt = DecisionTreeRegressor(
    max_depth=None,
    random_state=42,
)
dt.fit(x_train, y_train)
y_val_pred_dt = dt.predict(x_val)
metrics_dt = report_model("DecisionTree (baseline)", y_val, y_val_pred_dt)

[DecisionTree (baseline)]  MAE: 1,786.19 | R²: 0.8924


<a id="sec-5-compare"></a>
## 5. Model Comparison


In [71]:
# Compare baselines & pick current best
cmp = pd.DataFrame([
    {"model": "LinearRegression", **metrics_lin},
    {"model": "Ridge", **metrics_ridge},
    {"model": "Lasso", **metrics_lasso},
    {"model": "ElasticNet", **metrics_elasticnet},
    {"model": "RandomForest", **metrics_rf},
    {"model": "DecisionTree", **metrics_dt},
]).sort_values(by="MAE")

display(cmp)

best_name = cmp.iloc[0]["model"]
print(f"Current best (validation): {best_name}")


,model,MAE,R2
4,RandomForest,1317.901718,0.939171
5,DecisionTree,1786.185690,0.892430
2,Lasso,2795.819626,0.794446
1,Ridge,2795.883589,0.794445
0,LinearRegression,2796.087504,0.794445
3,ElasticNet,2960.628621,0.768348


Current best (validation): RandomForest


<a id="sec-6-tuning"></a>
## 6. Hyperparameter Tuning & Selection


In [72]:
param_grid = {
    "n_estimators": [200, 400, 600],
    "max_depth": [None, 20, 40],
    "max_features": ["sqrt", "log2", 0.5, 1],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

grid = list(ParameterGrid(param_grid))
n_total = len(grid)

best_score = float("inf")
best_params = None

for i, params in enumerate(grid, start=1):
    print(f"[{i}/{n_total}] Fitting with params: {params}")
    model = RandomForestRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(x_train, y_train_flat)
    y_pred = model.predict(x_val)
    mae = mean_absolute_error(y_val_flat, y_pred)

    print(f"    -> MAE: {mae:.3f}")

    if mae < best_score:
        best_score = mae
        best_params = params
        print(f"   New best params: {best_params} (MAE={best_score:.3f})")

print("\nBest params:", best_params)
print("Best MAE:", best_score)


[1/324] Fitting with params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
    -> MAE: 1342.210
   New best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200} (MAE=1342.210)
[2/324] Fitting with params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400}
    -> MAE: 1338.465
   New best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 400} (MAE=1338.465)
[3/324] Fitting with params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 600}
    -> MAE: 1337.612
   New best params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 600} (MAE=1337.612)
[4/324] Fitting with params: {'max_depth': None, 'max_features': 'sqrt', 

<a id="sec-7-submission"></a>
## 7. Submission File


In [74]:
# Build the best model (fit on train+val before predicting test)
y_train_flat = y_train.iloc[:, 0].values
y_val_flat = y_val.iloc[:, 0].values
# 2) Use the random forest model with the best hyperparameters found
best_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
best_model.fit(pd.concat([x_train, x_val], axis=0), pd.concat([pd.Series(y_train_flat), pd.Series(y_val_flat)], axis=0))

RandomForestRegressor(max_features=0.5, min_samples_split=5, n_estimators=600,
                      n_jobs=-1, random_state=42)

In [75]:
# Create Kaggle Submission (carID, price)

# Load the raw test file to obtain carID
test_raw_path = os.path.join(data_dir, "test.csv")
test_raw = pd.read_csv(test_raw_path)

#assert len(test_raw) == len(x_test), "Length mismatch between test.csv and x_test_final!"

# Determine which rows survived preprocessing
# Usually by using the index
surviving_ids = x_test.index

# Subset original raw test to surviving ids
test_raw_survived = test_raw.iloc[surviving_ids] # Mismatch: some surviving rows not found in raw test!

# Sanity check
assert len(test_raw_survived) == len(x_test)

# Predict on feature-selected test matrix
y_test_pred = best_model.predict(x_test)

# (Optional) Post-process predictions: clip negatives and round
# Here we round to whole units, as in the sample, without allowing negative prices:
y_test_pred = np.clip(y_test_pred, a_min=0, a_max=None)
y_test_pred_rounded = np.rint(y_test_pred).astype(int)

# Build the submission DataFrame
submission = pd.DataFrame({
    "carID": test_raw_survived["carID"],
    "price": y_test_pred_rounded  # optionally switch to y_test_pred if floats are allowed or preferred
})

# Save
sub_dir = os.path.join(data_dir, "submissions")
os.makedirs(sub_dir, exist_ok=True)
ts = datetime.now().strftime("%Y%m%d_%H%M")
sub_path = os.path.join(sub_dir, f"30_submission_{ts}.csv")
submission.to_csv(sub_path, index=False)

print(f"Submission saved to: {sub_path}")
display(submission.head(10))


Submission saved to: ../data/submissions\30_submission_20251126_0030.csv


,carID,price
0,89856,10446
1,106581,23772
2,80886,13898
3,100174,17184
4,81376,21914
5,85391,10408
6,82175,14971
7,95250,15640
8,85071,5316
9,96210,17246


<a id="sec-8-kaggle"></a>
## 8. Kaggle Submission

Set up your Kaggle API key in `.env` as `KAGGLE_USERNAME` and `KAGGLE_KEY`. The cell below submits the generated CSV and fetches submission status.


In [76]:
env_path = Path("..") / ".env"   
load_dotenv(env_path, override=True)

kuser = "joaoramos0610"  #os.getenv("KAGGLE_USERNAME")
kkey  = "96d0f23e975395a698b342e08e69bfc1" #os.getenv("KAGGLE_KEY")

print("KAGGLE_USERNAME:", kuser)
print(".env loaded from:", env_path.resolve())


KAGGLE_USERNAME: joaoramos0610
.env loaded from: C:\Users\jdiog\Desktop\NovaIMS\ML\repo\MachineLearningProject-NOVAIMS2025\.env


In [77]:
!kaggle competitions submit -c cars4you -f {sub_path} -m "Group 21 submission"


Successfully submitted to Cars4you



  0%|          | 0.00/415k [00:00<?, ?B/s]
  4%|▍         | 16.0k/415k [00:00<00:02, 145kB/s]
 96%|█████████▋| 400k/415k [00:00<00:00, 2.23MB/s]
100%|██████████| 415k/415k [00:00<00:00, 460kB/s] 


In [ ]:
# Get the results of the submission
#!kaggle competitions submissions -c cars4you

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\jdiog\anaconda3\Scripts\kaggle.exe\__main__.py", line 7, in <module>
    sys.exit(main())
             ~~~~^^
  File "C:\Users\jdiog\anaconda3\Lib\site-packages\kaggle\cli.py", line 70, in main
    out = args.func(**command_args)
  File "C:\Users\jdiog\anaconda3\Lib\site-packages\kaggle\api\kaggle_api_extended.py", line 1467, in competition_submissions_cli
    submissions = self.competition_submissions(
        competition, page_number=page, page_token=page_token, page_size=page_size
    )
TypeError: KaggleApi.competition_submissions() got an unexpected keyword argument 'page_number'


<a id="sec-9-error"></a>
## 9. Error Analysis


We had some predictions and results for our dataset. To improve further we want to find out, which entries are hard to predict and why.
For that we will try to analyze the absolute error for the validation dataset. For that we predict our random forest model on the validation set and calculate the absolute error. Then we merge it with the original validation data to have all features available for analysis. We look at the top entries with the highest absolute error.

In [83]:


# Build a new model on train data
final_model = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
final_model.fit(x_train, y_train)
# Predict on validation set
y_val_pred_analysis = final_model.predict(x_val)
# Calculate absolute error
abs_error = np.abs(y_val.values.ravel() - y_val_pred_analysis)

c:\Users\jdiog\anaconda3\envs\anaconda-2025.06-py3.11\Lib\site-packages\sklearn\base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [84]:
# Create a dataset for analysis which consists of carID, true values, predicted value and absolute error
analysis_df = x_val.copy()
analysis_df['true_values'] = y_val
analysis_df['predicted_values'] = y_val_pred_analysis
analysis_df['absolute_error'] = abs_error

# print shape of analysis_df
print(f"analysis_df shape: {analysis_df.shape}")

# display the analysis dataframe
display(analysis_df.head(10))

analysis_df shape: (18617, 28)


,Brand_Audi,Brand_BMW,Brand_Ford,Brand_Hyundai,Brand_Opel,Brand_Toyota,Brand_Škoda,fuelType_Electric,transmission_Automatic,transmission_Manual,...,previousOwners,mpg_diff_transmission,car_age,efficiency_ratio,mileage_per_year,previousOwners_sq,engine_tax_ratio,true_values,predicted_values,absolute_error
CarID,,,,,,,,,,,,,,,,,,,,,
7168,0.084507,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-1.0,0.0,...,-1.0,-0.409938,-0.666667,-0.579913,-0.848555,-0.500,0.020010,32480.0,35236.464492,2756.464492
64731,-0.464789,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.0,0.987578,-0.333333,1.630936,0.835234,-0.500,-0.292139,9995.0,9015.721974,979.278026
43669,1.605634,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,-0.962733,0.000000,-0.202639,0.546269,1.500,-0.277024,10159.0,10425.070655,266.070655
2212,-0.507042,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,-0.391304,0.000000,0.137647,0.171164,1.500,-0.161277,10400.0,10168.988993,231.011007
23000,1.563380,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,-0.391304,0.000000,0.430133,1.152461,1.500,-0.164155,12995.0,11875.827073,1119.172927
57658,0.591549,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-1.0,1.0,...,-1.0,0.677019,0.333333,-0.067370,-0.488776,-0.500,2.261725,13995.0,15922.628556,1927.628556
27991,-0.380282,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-1.0,0.0,...,-1.0,0.316770,1.000000,0.286032,0.701248,-0.500,1.692524,6275.0,7058.474961,783.474961
69358,-0.676056,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-1.0,0.074534,0.333333,0.415054,-0.133030,-0.500,1.407924,17495.0,16122.663167,1372.336833
42126,-0.014085,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.5,-0.167702,-0.666667,0.692797,0.027057,-0.375,-0.282135,15995.0,15839.681633,155.318367


In [88]:
# Load the raw train data and merge with analysis_df to get car details
train_raw_path = os.path.join(data_dir, "train.csv")
train_raw = pd.read_csv(train_raw_path)

# Merge using index from analysis_df and 'carID' column from train_raw
analysis_df = analysis_df.merge(train_raw, left_index=True, right_on='carID', how='left')

# Optional: set carID as index again if needed
analysis_df.set_index('carID', inplace=True)

display(analysis_df.head(10))

,Brand_Audi,Brand_BMW,Brand_Ford,Brand_Hyundai,Brand_Opel,Brand_Toyota,Brand_Škoda,fuelType_Electric,transmission_Automatic,transmission_Manual,...,price,transmission,mileage_y,fuelType,tax_y,mpg_y,engineSize_y,paintQuality%_y,previousOwners_y,hasDamage
carID,,,,,,,,,,,,,,,,,,,,,
7168,0.084507,0.0,0.0,0.0,1.0,0.0,0.0,1.0,-1.0,0.0,...,11000,Manual,29526.000000,Petrol,0.0,67.3,1.0,NaN,3.0,0.0
64731,-0.464789,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,10690,Manual,36767.000000,Petrol,145.0,NaN,1.4,83.0,1.0,0.0
43669,1.605634,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,21998,Semi-Auto,25313.000000,Diesel,160.0,51.4,3.0,30.0,2.0,0.0
2212,-0.507042,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,20990,Semi-Auto,20630.000000,Diesel,145.0,57.7,2.0,42.0,2.0,0.0
23000,1.563380,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14500,Automatic,18460.000000,Diesel,125.0,56.6,1.5,39.0,4.0,0.0
57658,0.591549,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-1.0,1.0,...,12791,Manual,48771.000000,Diesel,150.0,54.3,2.0,98.0,4.0,0.0
27991,-0.380282,0.0,0.0,1.0,0.0,0.0,0.0,1.0,-1.0,0.0,...,13250,Manual,32643.000000,Diesel,NaN,NaN,2.0,90.0,1.0,0.0
69358,-0.676056,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5998,Manual,39296.000000,Petrol,125.0,51.4,1.2,57.0,3.0,NaN
42126,-0.014085,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,25980,Semi-Auto,8999.000000,Petrol,145.0,45.6,1.5,82.0,3.0,0.0
